# GANForge: Custom DCGAN Fine-Tuning on Rangoli Art
### Google Colab GPU Accelerated Training Pipeline (P40)

This notebook executes the end-to-end training of the **Deep Convolutional Generative Adversarial Network (DCGAN)** on Google Colab's cloud GPU (NVIDIA T4 / V100).

**Hardware Recommendation:**
1. Go to menu: **Runtime** -> **Change runtime type**
2. Hardware accelerator: Select **T4 GPU** (or any available GPU)
3. Click **Save**

In [ ]:
# Step 1: Verify Hardware Accelerator
!nvidia-smi

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n[+] TensorFlow successfully detected GPU: {gpus}")
else:
    print("\n[-] Warning: No GPU detected. Check Runtime -> Change runtime type -> T4 GPU.")

### Step 2: Environment Setup & Code Sync

In [ ]:
# Clone or pull latest GANForge repository
!git clone https://github.com/Samar/GANForge.git || (cd GANForge && git pull)
%cd GANForge
!ls -la

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
print("[+] Dependencies installed successfully!")

### Step 3: Dataset Ingestion & Preprocessing (Unit 1)
Validates, normalizes to $[-1, 1]$, and splits the 300-image Rangoli dataset into 80% train and 20% validation.

In [ ]:
!python -m data.prepare_rangoli_dataset --count 300

### Step 4: Run DCGAN Training Loop (50 Epochs)
Executes custom `tf.GradientTape` training loop, generating sample grids at epochs 1, 10, 25, 50.

In [ ]:
!python -m training.train --epochs 50 --batch-size 32

### Step 5: Visualize Loss Curves & Convergence

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.read_csv("outputs/metrics/training_history.csv")

plt.figure(figsize=(10, 5))
plt.plot(history_df['epoch'], history_df['generator_loss'], label='Generator Loss', color='royalblue', lw=2)
plt.plot(history_df['epoch'], history_df['discriminator_loss'], label='Discriminator Loss', color='crimson', lw=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (BCE)', fontsize=12)
plt.title('GANForge - DCGAN Adversarial Loss Dynamics (50 Epochs)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("outputs/metrics/loss_curve.png", dpi=150)
plt.show()

### Step 6: Inspect Milestone Sample Grids

In [ ]:
from PIL import Image
import glob

sample_files = sorted(glob.glob("outputs/samples/epoch_*_grid.png"))
for f in sample_files:
    print(f"Displaying: {f}")
    display(Image.open(f))

### Step 7: Bundle & Download Outputs for Local Streamlit Dashboard

In [ ]:
from google.colab import files
!zip -r ganforge_colab_outputs.zip outputs/ data/processed/
files.download('ganforge_colab_outputs.zip')
print("[+] Artifacts packaged and download triggered!")